# Logjam — Worked Examples

This notebook mirrors the worked examples in the [Logjam README](https://github.com/mgkay/Logjam). Examples are ordered for incremental capability building — each one introduces a focused set of Logjam functions that later examples build upon. The full capability set is described in the README overview.

**Examples:**
1. [Spatial Data](#example-1--spatial-data)
2. [Mapping](#example-2--mapping)
3. [Facility Location](#example-3--facility-location)
4. [Pickup and Delivery Routing](#example-4--pickup-and-delivery-routing)
5. [Vehicle Routing (OSM Network)](#example-5--vehicle-routing-osm-network)

---

## Example 1 — Spatial Data

Logjam includes a built-in U.S. gazetteer covering cities (`usplace`), counties (`uscounty`), 3- and 5-digit ZIP codes (`uszcta3`, `uszcta5`), CBSAs (`uscbsa`), and CSAs (`uscsa`). All tables share a consistent schema: FIPS codes for joining, LON/LAT coordinates following the Logjam convention (longitude first), and population counts where available. This example demonstrates loading and filtering place data, FIPS code conversion, and geocoding.

In [2]:
using Logjam
using DataFrames

# Load U.S. place data (cities, towns, CDPs) and inspect the schema
places = usplace()
display(first(places[:, [:NAME, :ST, :LON, :LAT, :POP, :ISCUS]], 4))

Row,NAME,ST,LON,LAT,POP,ISCUS
,String,Symbol,Float64,Float64,Int64,Bool
1,Albertville,AL,-86.2107,34.2631,22386,true
2,Alexander City,AL,-85.9371,32.9272,14843,true
3,Anniston,AL,-85.8109,33.6735,21564,true
4,Auburn,AL,-85.4895,32.6077,76143,true


In [ ]:
# FIPS conversion: state symbol ↔ FIPS code
fips_nc = st2fips(:NC)           # 37
fips2st(fips_nc)                 # :NC

# Filter to NC cities with population over 100,000
nc_large = filter(r -> r.STFIP == fips_nc && r.POP > 100_000, places)
println("$(nrow(nc_large)) NC cities with pop > 100k")

# Forward geocoding: city name to coordinates
geo = loc2lonlat("Raleigh", state=:NC)
println("Raleigh: ($(round(geo.lon; digits=2)), $(round(geo.lat; digits=2))), source=$(geo.source)")

# Reverse geocoding: coordinates to nearest named place
nearest = lonlat2loc(-78.85, 35.73, filter(r -> r.POP > 50_000 && r.ISCUS, places))
println("Nearest large city: $(nearest.desc)")

# Load continental US 3-digit ZIP centroids
z3 = filter(r -> r.ISCUS, uszcta3())
println("Continental US 3-digit ZIPs: $(nrow(z3))")

**After-action.** The `ISCUS` flag is the standard filter for continental U.S. analysis — it removes Alaska, Hawaii, Puerto Rico, and other territories that would distort national maps. All coordinate columns follow the (LON, LAT) convention throughout Logjam, which means western longitudes are negative. The `st2fips` and `fips2st` functions enable joins between datasets that use different geographic identifiers. `loc2lonlat` provides forward geocoding — from place names, postal codes, or street addresses to coordinates — and returns source attribution and distance uncertainty along with each result. `lonlat2loc` is the reverse: given a coordinate, it finds the nearest named place and reports whether the point is "in" the place (using an area-based radius) or a distance and direction away; it is used in later examples to label facility hub coordinates with interpretable city names.

---

## Example 2 — Mapping

Logjam's `makemap` function creates GeoMakie map figures with automatic projection, region detection, and a FAF5 interstate highway background. This example maps NC cities and introduces the full mapping toolkit: `makemap` for the figure and axis, `scatter!` and `text!` for data, `aligntext` for automatic label positioning, and the `hborders` return value for customizing the built-in road overlay.

In [ ]:
using Logjam
using CairoMakie, GeoMakie, DataFrames

# Load NC cities with population > 100,000
cities = filter(r -> r.STFIP == st2fips(:NC) && r.POP > 100_000, usplace())
x, y, name = cities.LON, cities.LAT, cities.NAME

# Create map — auto-fits region to data; draws FAF5 interstates by default
fig, ax, hb = makemap(x, y)
ax.title = "North Carolina Cities with Population > 100,000"

hb[1].color[] = (:steelblue, 0.5)

# Plot city locations and labels
scatter!(ax, x, y, color=:red, markersize=12)
text!(ax, x, y, text=name; aligntext(x, y)...)

dcf()

**After-action.** `makemap` detects when coordinates fall within the continental U.S. and automatically overlays the FAF5 interstate network as a geographic reference. The `hborders` return value exposes the road and border line handles for post-hoc styling, as shown above for the interstate color. `aligntext(x, y)` returns a named tuple of keyword arguments (`:align`, `:offset`) that are splatted into `text!` via `...`, automatically positioning each label to avoid overlap with its marker.

---

## Example 3 — Facility Location

A classic strategic logistics problem: place a fixed number of distribution hubs to minimize total weighted distance to customers. Using U.S. 3-digit ZIP code centroids as demand points and population as demand weight, the *p*-median model selects six hubs that minimize total people-miles. This example introduces `dists` for distance matrix construction, `pmedian`, `alloclines`, and `lonlat2loc` for the full facility location workflow, and `prt` for formatted matrix display.

In [5]:
using Logjam
using CairoMakie, GeoMakie, DataFrames

# Load continental US 3-digit ZIP code centroids
z3 = filter(r -> r.ISCUS, uszcta3())
XY = hcat(z3.LON, z3.LAT)

# Build population-weighted cost matrix: C[i,j] = distance(i,j) × population(j)
C = dists(XY, XY, :mi) .* z3.POP'

882×882 Matrix{Float64}:
 0.0        1.27581e6  4.80354e6  2.18512e6  …  9.2409e8   4.91286e7
 3.51217e6  0.0        5.41746e6  2.77588e6     9.257e8    4.92168e7
 1.77188e7  7.25901e6  0.0        3.21568e6     9.08958e8  4.8294e7
 1.24675e7  5.75327e6  4.974e6    0.0           9.21586e8  4.89931e7
 2.25424e7  8.77433e6  9.50444e6  2.97611e6     9.3507e8   4.97385e7
 1.87714e7  6.75069e6  9.60909e6  3.61903e6  …  9.38799e8  4.99417e7
 1.85466e7  6.78733e6  9.43232e6  3.40518e6     9.3798e8   4.98968e7
 2.73191e7  9.9899e6   1.16637e7  4.61751e6     9.44071e8  5.0234e7
 3.57961e7  1.33727e7  1.33056e7  5.41801e6     9.45415e8  5.0311e7
 4.18636e7  1.54333e7  1.51848e7  6.66314e6     9.51715e8  5.06587e7
 3.45029e7  1.23299e7  1.3883e7   6.19699e6  …  9.51841e8  5.06621e7
 3.6616e7   1.33181e7  1.4167e7   6.16469e6     9.5125e8   5.0631e7
 3.60784e7  1.31345e7  1.40081e7  6.05778e6     9.50721e8  5.06018e7
 ⋮                                           ⋱  ⋮          
 1.13975e9  4.14776e8 

In [ ]:
# Solve p-median: select 6 hubs minimizing total people-miles
y, TC, W = pmedian(6, C; verbose=false)

# Reverse-geocode hub coordinates to nearest large city
hubs = lonlat2loc(XY[y, :], filter(r -> r.POP >= 50_000 && r.ISCUS, usplace()))

# Display hub locations (prt auto-formats coordinates)
prt(XY[y, :]; rows=hubs.name, cols=["LON", "LAT"], row_title="Hubs")

In [ ]:
# Map: continental US with allocation lines and hub markers
fig, ax = makemap(region=:CUS)

colors = Makie.wong_colors()[1:6]
X, Y = alloclines(W, XY, XY)
for (ci, i) in enumerate(y)
    lines!(ax, X[i], Y[i], color=(colors[ci], 0.2), linewidth=0.8)
end

scatter!(ax, XY[:, 1], XY[:, 2], color=:red, markersize=3, strokewidth=0)
hub_xy = XY[y, :]
scatter!(ax, hub_xy[:, 1], hub_xy[:, 2], color=colors, markersize=18,
         strokewidth=3, strokecolor=:white)
text!(ax, hub_xy[:, 1], hub_xy[:, 2], text=hubs.name; aligntext(hub_xy[:, 1], hub_xy[:, 2])...)

ax.title = "Optimal Facility Locations (P-Median)\nSelected from 3-Digit ZIP Code Centroids (Population-Weighted)"
dcf()

**After-action.** `dists(XY, XY, :mi)` computes a full pairwise great-circle distance matrix in miles; the `:mi` symbol selects miles, `:km` selects kilometers, and `:gc` returns dimensionless radians. Broadcasting `.* z3.POP'` weights each column by the destination's population, converting the distance matrix into a cost matrix in people-miles. `pmedian` returns the hub indices `y`, total cost `TC`, and the allocation matrix `W` (a sparse indicator mapping each demand point to its nearest hub). `alloclines` converts `W` into NaN-separated line segment vectors per hub, which `lines!` renders efficiently without a loop per connection. `lonlat2loc` reverse-geocodes the hub coordinates to the nearest large city, providing interpretable labels. `prt` displays the hub coordinate matrix as a formatted table with city-name row labels, automatic decimal detection, and comma-separated large numbers. For problems where the number of facilities is itself a decision, Logjam provides UFL heuristics — `ufladd`, `ufldrop`, `uflxchg`, and `ufl` — that optimize both facility selection and count.

---

## Example 4 — Pickup and Delivery Routing

Road network routing in Logjam uses the FAF5 national freight highway network as the foundation. The standard pipeline — `cropnetwork` → `addconnectors` → `links2graph` → `shortestpaths` — prepares a network for any routing task.

A pickup and delivery problem (PDP) is a routing problem where each shipment has a distinct origin and destination. This example routes five shipments across ten NC cities using the FAF5 highway network. The vehicle does not return to a starting depot — the route connects pickups and deliveries in the optimal sequence.

In [ ]:
using Logjam
using CairoMakie, GeoMakie, DataFrames

# Load NC cities with population > 100,000
cities = filter(r -> r.STFIP == st2fips(:NC) && r.POP > 100_000, usplace())

# City index (alphabetical order from filter)
prt(DataFrame(Index = 1:nrow(cities), City = cities.NAME))

In [ ]:
# Define 5 shipments with distinct origins and destinations
sh = DataFrame(                  # shipments
    b = [2, 3, 10, 7, 6],   # Pickup city index
    e = [8, 4,  1, 5, 9]    # Delivery city index
)

# Shipment manifest with city names
prt(DataFrame(
    Shipment = 1:nrow(sh),
    Origin = cities.NAME[sh.b],
    Destination = cities.NAME[sh.e]))

In [ ]:
# Build road network: crop FAF5 to region, attach city connectors
x, y = cities.LON, cities.LAT
dfN, dfL = addconnectors(cropnetwork(faf5nodes(), faf5links(), x, y)..., x, y)  # nodes, links
D, P = shortestpaths(links2graph(dfL), nrow(cities))  # distance, parents

# Display highway distance matrix (miles) with abbreviated city names
cnames = [length(n) > 7 ? first(n, 4) * "." : n for n in cities.NAME]
prt(round.(Int, D); rows=cnames, cols=cnames, row_title="City")

# Construct route with savings heuristic, then improve with 2-opt
rteTCh(r) = rteTC(r, sh, D)  # route total cost handle
rte = savings(rteTCh, sh)    # route
rte, cost = twoopt(rte[1], rteTCh)
println("Route cost: $(round(cost; digits=2)) miles")

In [ ]:
# Map: NC region with road network overlay and optimized route
fig, ax = makemap(x, y)
plotroads!(ax, dfN, dfL)

plotroute!(ax, rte, sh, P, dfN; color=:red, linewidth=2.5, show_markers=false)

scatter!(ax, x, y, color=:blue, markersize=10)
text!(ax, x, y, text=cities.NAME; aligntext(x, y)...)

ax.title = "Multi-Stop PDP: Savings + 2-Opt\n(5 Shipments, 10 NC Cities, FAF5 Network)"
dcf()

**After-action.** `cropnetwork` extracts the FAF5 subgraph covering the demand points, reducing graph size before solving. `addconnectors` appends connector edges from each demand point to its nearest network node. `shortestpaths` computes shortest-path distances `D` and parent pointers `P` from the connector nodes. `prt` displays the distance matrix with city-name rows/columns and the `row_title` keyword in the upper-left corner; it also generates the city index and shipment manifest from inline DataFrames. `plotroads!` renders the road network with FCLASS-based styling — roads are colored and sized by functional class (interstates in muted blue, arterials in warm yellow, local roads in white/gray). `savings` constructs an initial route by iteratively merging the most cost-saving shipment pair, then `twoopt` improves it by reversing sub-sequences. `plotroute!` reconstructs the road-following path for each leg from the parent pointers and renders the result.

---

## Example 5 — Vehicle Routing (OSM Network)

> **Requires OSM extension** (`LightOSM`, `NearestNeighbors`) **and Nominatim extension** (`HTTP`, `JSON3`).

The `osm_roads` function downloads a local OpenStreetMap road network and integrates it with the same connector pipeline used for FAF5 routing. This example solves a capacitated multi-vehicle VRP in Gainesville, FL — three vehicles with a maximum of three deliveries each share nine stops from a central depot, with each vehicle returning after completing its route. Stop locations are specified as street addresses and geocoded to coordinates using `loc2lonlat`.

In [ ]:
using LightOSM, NearestNeighbors  # OSM extension
using HTTP, JSON3                  # Nominatim geocoding extension
using Logjam
using CairoMakie, GeoMakie, DataFrames

# Stop addresses: depot (UF campus) + 9 delivery locations in Gainesville, FL
stops = DataFrame(
    STREET = ["1580 Stadium Rd", "1620 W University Ave", "3500 SW Archer Rd",
              "200 NW 13th St", "3000 NE Waldo Rd", "4001 NW 43rd St",
              "1800 SW 13th St", "1230 NE 23rd Ave", "2900 SW 34th St",
              "620 NW 8th Ave"],
    CITY = fill("Gainesville", 10),
    STATE = fill(:FL, 10)
)

# Geocode addresses to (lon, lat) — Nominatim with city/state fallback
gc = loc2lonlat(stops)
x, y = gc.LON, gc.LAT

# Download OSM road network covering the stop region
bb = mapbbox(x, y; xexpand=0.1, yexpand=0.1)[1]
dfN0, dfL0 = osm_roads((bb[1]..., bb[2]...); cache_dir=joinpath(@__DIR__, "data"))

# Build network and shortest paths
dfN, dfL = addconnectors(dfN0, dfL0, x, y; add_nf_nf=false)
D, P = shortestpaths(links2graph(dfL), length(x))  # distance, parents

In [ ]:
# VRP: all deliveries depart from depot (stop 1), max 3 per vehicle
sh = DataFrame(b = fill(1, 9), e = 2:10)  # shipments
tr = (b=[1], e=[1])                        # truck
rteTCh(r) = length(r) > 6 ? Inf : rteTC(r, sh, D, tr)  # route total cost handle

rte = savings(rteTCh, sh)  # route
rte = [twoopt(r, rteTCh)[1] for r in rte]
println(length(rte), " routes, ", [length(r)÷2 for r in rte], " stops each")

In [ ]:
# Map: OSM road network with color-coded vehicle routes
fig, ax = makemap(x, y)
plotroads!(ax, dfN, dfL)

plotroute!(ax, rte, sh, P, dfN; tr=tr, linewidth=2.5, show_markers=false)

scatter!(ax, x[2:end], y[2:end], color=:blue, markersize=12)
scatter!(ax, [x[1]], [y[1]], color=:green, markersize=16, marker=:rect)
text!(ax, [x[1]], [y[1]], text=["Depot"]; aligntext([x[1]], [y[1]])...)

ax.title = "Multi-Vehicle VRP: Savings + 2-Opt\n(9 Deliveries, 3 Vehicles, Gainesville FL)"
dcf()

**After-action.** `loc2lonlat` geocodes the stop addresses via Nominatim's structured query API; results are cached to CSV so subsequent runs skip the API calls. If an address cannot be resolved, the function falls back to the city/state centroid and flags the result as `PARTIAL`. The geocoded coordinates then feed into the standard OSM routing pipeline. `mapbbox` derives a bounding box with 10% expansion to ensure the downloaded OSM region covers all stops. `osm_roads` downloads drivable roads from the Overpass API and caches results as CSV; subsequent calls with the same bbox load from cache. `add_nf_nf=false` disables direct demand-to-demand connectors that would bypass the road network. The capacity constraint is enforced through the cost function: `length(r) > 6 ? Inf` limits each route to three deliveries (each shipment appears as a pickup–delivery pair), causing `savings` to produce multiple routes. `tr=(b=[1], e=[1])` specifies that each route begins and ends at the depot. The multi-route `plotroute!` method automatically assigns a distinct color per vehicle from the Wong palette. For scenarios requiring local OSM detail integrated with the national FAF5 network, `stitchnetworks` creates connector edges between the two networks, returning a unified graph compatible with the standard routing pipeline.